# trip_duration — Diagnóstico

Notebook de diagnóstico del modelo de **trip_duration**. No re-entrena: carga
el modelo `trip_duration@champion` desde el Model Registry de MLflow y lo
analiza sobre los datos de gold (`nessie.gold.yellow_taxi_trips`).

Secciones:
1. Setup (Spark + MLflow client).
2. Cargar el modelo registrado vía alias `@champion`.
3. Metadatos del run de origen (algoritmo, params, métricas).
4. Reconstruir los splits (`train` / `val` / `test_id` / `test_ood`) y medir el modelo en cada uno.
5. Predicciones sobre `test_ood` (2025 — drift probe).
6. Predicted vs Actual.
7. Residuales.
8. Importancia de features (agnóstica al algoritmo).
9. Drift — métricas por año sobre datos no usados para entrenar.
10. Conclusiones.

In [ ]:
import mlflow
import mlflow.spark
from mlflow import MlflowClient
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ml.common import (
    TEST_YEARS,
    get_ml_spark, setup_mlflow, make_splits,
)

## 1. Setup — Spark + MLflow

In [ ]:
EXPERIMENT = "trip_duration"
SOURCE = "nessie.gold.yellow_taxi_trips"
LABEL = "trip_duration_minutes"
YEAR_COL = "pickup_year"

spark = get_ml_spark("ml-trip-duration-diagnostics")
setup_mlflow(EXPERIMENT)
client = MlflowClient()
spark.version

## 2. Cargar el modelo desde el Model Registry

Cada ejecución de `trip_duration.py` registra una nueva versión del modelo
en el registry y reasigna el alias `@champion` a esa versión. Resolver el
alias da siempre el modelo más reciente, sin tener que rastrear run IDs.
Para inspeccionar versiones anteriores, sustituir `"champion"` por el número
de versión (`mv = client.get_model_version(EXPERIMENT, "3")`).

In [ ]:
mv = client.get_model_version_by_alias(name=EXPERIMENT, alias="champion")
source_run = client.get_run(mv.run_id)
print(f"Registered model: {EXPERIMENT} v{mv.version}")
print(f"Source run ID:    {mv.run_id}")
print(f"Run name:         {source_run.data.tags.get('mlflow.runName')}")
print(f"Algorithm:        {source_run.data.tags.get('algorithm')}")
print(f"Train years:      {source_run.data.tags.get('train_years')}")
print(f"Started:          {pd.Timestamp(source_run.info.start_time, unit='ms')}")
print(f"Description:")
print(f"  {mv.description or '(none)'}")

In [ ]:
# Métricas y parámetros del run de origen del modelo registrado
metrics_df = pd.DataFrame.from_dict(source_run.data.metrics, orient='index', columns=['value'])
params_df  = pd.DataFrame.from_dict(source_run.data.params,  orient='index', columns=['value'])
display(metrics_df.sort_index())
display(params_df.sort_index())

## 3. Cargar el `PipelineModel`

`mlflow.spark.load_model` deserializa el `PipelineModel` desde el bucket
`mlflow-artifacts` en MinIO. Las stages dependen del algoritmo ganador en
Phase 1: el caso típico (GBT / RF) usa `StringIndexer` + `VectorAssembler`
+ estimator; si gana LR, se añaden `OneHotEncoder` por categórica y
`StandardScaler` antes del estimador.

In [ ]:
model_uri = f"models:/{EXPERIMENT}@champion"
model = mlflow.spark.load_model(model_uri)
for i, stage in enumerate(model.stages):
    print(f"{i}: {type(stage).__name__}")

## 4. Reconstruir splits y medir cada uno

Usamos `make_splits` con la seed por defecto (la misma que el script de
entrenamiento), así los conjuntos `train` / `val` / `test_id` (estratificados
70 / 15 / 15 dentro de `TRAIN_YEAR_SUBSETS[-1]`) y `test_ood` (`TEST_YEARS` =
2025, íntegro) coinciden con los del run de origen. Antes de dividir
aplicamos el mismo filtro que el script de entrenamiento:
`trip_duration_minutes > 0` (silver garantiza `pickup < dropoff` pero
`unix_timestamp` trunca a segundos, así que viajes sub-segundo aparecen
como duración 0).

Comparar métricas entre splits cuenta dos historias distintas:

- **`val` ↔ `test_id`** — ambos in-distribution. Si `test_id` empeora mucho
  respecto a `val`, hay overfitting en la selección de modelo.
- **`test_id` ↔ `test_ood`** — `test_id` mide calidad del modelo,
  `test_ood` mezcla calidad + drift. La diferencia es el tamaño del drift.

In [ ]:
# Same pre-filter as the training script: trip_duration_minutes must be > 0.
df = spark.table(SOURCE).filter(col(LABEL) > 0)
train, val, test_id, test_ood = make_splits(df, YEAR_COL, TEST_YEARS)
splits = {"train": train, "val": val, "test_id": test_id, "test_ood": test_ood}
for s in splits.values():
    s.cache()

print("Split sizes:")
for name, s in splits.items():
    print(f"  {name:<10} {s.count():>12,} rows")

print("\nMetrics per split:")
print(f"  {'split':<10} {'rmse':>10} {'mae':>10} {'r2':>10}")
for name, s in splits.items():
    pred = model.transform(s).select(LABEL, "prediction")
    rmse = RegressionEvaluator(labelCol=LABEL, predictionCol="prediction", metricName="rmse").evaluate(pred)
    mae  = RegressionEvaluator(labelCol=LABEL, predictionCol="prediction", metricName="mae" ).evaluate(pred)
    r2   = RegressionEvaluator(labelCol=LABEL, predictionCol="prediction", metricName="r2"  ).evaluate(pred)
    print(f"  {name:<10} {rmse:>10.4f} {mae:>10.4f} {r2:>10.4f}")

## 5. Predicciones sobre `test_ood`

Tomamos una muestra del 5 % de `test_ood` (2025) para llevarla a pandas y
poder dibujar. Las métricas globales ya están en la sección 4. Sustituir
`test_ood` por `test_id` en este bloque para ver el comportamiento
in-distribution.

In [ ]:
sample_fraction = 0.05
test_pred_full = model.transform(test_ood).select(LABEL, "prediction")
test_pdf = test_pred_full.sample(sample_fraction, seed=42).toPandas()
test_pdf["residual"] = test_pdf[LABEL] - test_pdf["prediction"]
print(f"test_ood sample: {len(test_pdf):,} rows ({sample_fraction:.0%} of test_ood)")
test_pdf.describe()

## 6. Predicted vs Actual

Si el modelo está bien calibrado, los puntos se alinean cerca de la diagonal
`y = x`. Desviaciones sistemáticas (curvatura, sesgo) revelan miscalibración.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(test_pdf[LABEL], test_pdf["prediction"], alpha=0.08, s=6)
lim = float(max(test_pdf[LABEL].max(), test_pdf["prediction"].max()))
ax.plot([0, lim], [0, lim], "r--", linewidth=1, label="y = x")
ax.set_xlabel(f"actual {LABEL}")
ax.set_ylabel(f"predicted {LABEL}")
ax.set_title(f"Predicted vs Actual — {EXPERIMENT} (test_ood sample)")
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.legend()
plt.tight_layout()

## 7. Residuales

Dos vistas:
- **Residual vs predicción** — bandas horizontales = error homoscedástico (bueno);
  forma de embudo = heteroscedasticidad (el error crece con la predicción).
- **Histograma de residuales** — debería estar centrado en 0 y simétrico.
  Asimetría revela sesgo (el modelo sobre/subestima sistemáticamente).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(test_pdf["prediction"], test_pdf["residual"], alpha=0.08, s=6)
axes[0].axhline(0, color="r", linestyle="--", linewidth=1)
axes[0].set_xlabel("predicted")
axes[0].set_ylabel("residual (actual − predicted)")
axes[0].set_title("Residuales vs predicción")

axes[1].hist(test_pdf["residual"], bins=80, edgecolor="black", linewidth=0.3)
axes[1].axvline(0, color="r", linestyle="--", linewidth=1)
axes[1].set_xlabel("residual")
axes[1].set_ylabel("frequency")
axes[1].set_title("Distribución de residuales")

plt.tight_layout()

## 8. Importancia de features

Calculamos importancia / coeficientes según el algoritmo ganador:

- **GBT / RF** — `featureImportances` (reducción media de impureza ponderada).
- **LR** — magnitud absoluta de los coeficientes. Las features están
  estandarizadas vía `StandardScaler` en su pipeline, así que comparar
  magnitudes tiene sentido.

Si el ganador usa OneHotEncoder (caso LR), el vector resultante es más ancho
que las columnas de entrada al `VectorAssembler`, así que se muestra por
índice. Si las longitudes coinciden (StringIndexer-only, caso GBT / RF) cada
importancia se asocia a su columna original.

In [ ]:
final = model.stages[-1]
algorithm = type(final).__name__

asm = next((s for s in model.stages if isinstance(s, VectorAssembler)), None)
inputs = list(asm.getInputCols()) if asm is not None else None

if hasattr(final, "featureImportances"):
    values = final.featureImportances.toArray()
    metric_name = "importance"
elif hasattr(final, "coefficients"):
    values = abs(final.coefficients.toArray())
    metric_name = "abs_coef"
else:
    raise RuntimeError(f"No featureImportances / coefficients available for {algorithm}")

label_col = "feature" if inputs is not None and len(inputs) == len(values) else "feature_idx"
labels = inputs if label_col == "feature" else list(range(len(values)))

fi = (
    pd.DataFrame({label_col: labels, metric_name: values})
    .sort_values(metric_name, ascending=False)
    .reset_index(drop=True)
)
# OHE pipelines pueden producir cientos de dimensiones; mostramos top 30.
top = fi.head(30)
display(top)

fig, ax = plt.subplots(figsize=(8, max(3, 0.3 * len(top))))
sns.barplot(data=top, x=metric_name, y=label_col, ax=ax, color="steelblue")
ax.set_title(f"Top feature importances — {EXPERIMENT} ({algorithm})")
plt.tight_layout()

## 9. Drift — métricas por año

Calculamos RMSE / MAE / R² por año **sobre los datos no usados para
entrenar** (`val ∪ test_id ∪ test_ood`). Esto da una lectura limpia de
cómo generaliza el modelo en cada año sin contaminación por overfitting:
para los años de entrenamiento (`TRAIN_YEAR_SUBSETS[-1]`) evaluamos
sobre los splits `val` + `test_id` de ese año; para 2025 evaluamos
sobre `test_ood`.

Una diferencia notable entre 2025 y los años de entrenamiento es la
firma de **drift** (post-COVID, tarifa CBD). Métricas similares = drift
despreciable.

In [ ]:
unseen = val.unionByName(test_id).unionByName(test_ood)
all_pred = model.transform(unseen).select(YEAR_COL, LABEL, "prediction").cache()
years = sorted(r[YEAR_COL] for r in all_pred.select(YEAR_COL).distinct().collect())

rows = []
for y in years:
    yr = all_pred.filter(col(YEAR_COL) == y)
    n = yr.count()
    if n == 0:
        continue
    eval_kwargs = dict(labelCol=LABEL, predictionCol="prediction")
    rows.append({
        "year": y,
        "n":    n,
        "rmse": RegressionEvaluator(metricName="rmse", **eval_kwargs).evaluate(yr),
        "mae":  RegressionEvaluator(metricName="mae",  **eval_kwargs).evaluate(yr),
        "r2":   RegressionEvaluator(metricName="r2",   **eval_kwargs).evaluate(yr),
    })
all_pred.unpersist()

drift = pd.DataFrame(rows)
display(drift)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
test_year_set = set(TEST_YEARS)
for ax, metric in zip(axes, ("rmse", "mae", "r2")):
    colors = ["#d62728" if y in test_year_set else "#1f77b4" for y in drift["year"]]
    ax.bar(drift["year"].astype(str), drift[metric], color=colors)
    ax.set_xlabel("year")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} per year (red = test)")
plt.tight_layout()

## 10. Conclusiones

> _Rellenar con la lectura cualitativa del modelo:_
>
> - ¿`val` ↔ `test_id` revelan overfitting en la selección de hiperparámetros? (sección 4)
> - ¿Cuánto más alto es el error en `test_ood` (2025) que en `test_id`? ¿Drift apreciable? (sección 4)
> - ¿El modelo está bien calibrado sobre test_ood? (sección 6)
> - ¿Hay heteroscedasticidad o sesgo en los residuales? (sección 7)
> - ¿Qué features dominan? ¿Tiene sentido? (sección 8)
> - ¿La degradación por año es gradual o un salto solo en 2025? (sección 9)
> - Próximos pasos / mejoras.